In [ ]:
# ============================================================
# CELL 1: Install ONLY what's missing (no bitsandbytes)
# Run once, then restart kernel
# ============================================================
import sys
!{sys.executable} -m pip install --no-cache-dir sentencepiece --no-deps

In [ ]:
# ============================================================
# CELL 2: Imports & Sanity Check
# ============================================================
import torch
import re
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM      : {total_vram:.1f} GB")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

SMALL_MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
TEACHER_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

NUM_PROBLEMS   = 50      # start small to verify, bump to 300 for final run
N_VALUES       = [1, 3, 5, 10, 20]
# NUM_PROBLEMS   = 10      # start small to verify, bump to 300 for final run
# N_VALUES       = [1, 5, 10]
TEMPERATURE    = 0.7
MAX_NEW_TOKENS = 512

print(f"Small model  : {SMALL_MODEL_ID}")
print(f"Teacher model: {TEACHER_MODEL_ID}")
print(f"Problems     : {NUM_PROBLEMS}")
print(f"N values     : {N_VALUES}")

In [ ]:
import accelerate
print(accelerate.__version__)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
os.environ["HF_TOKEN"] = "token"

from huggingface_hub import login
login(token="token")

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    token="token"
)

In [ ]:
# ============================================================
# CELL 4: Load GSM8K Dataset
# ============================================================

import json, urllib.request, random, re

def load_gsm8k(n_problems=50, seed=42):
    url = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"
    print("Loading GSM8K from GitHub...")
    
    problems = []
    with urllib.request.urlopen(url) as f:
        for line in f:
            row = json.loads(line.decode('utf-8'))
            match = re.search(r'####\s*([\d,]+)', row['answer'])
            if match:
                problems.append({
                    'question'    : row['question'],
                    'full_answer' : row['answer'],
                    'ground_truth': match.group(1).replace(',', '')
                })
    
    random.seed(seed)
    random.shuffle(problems)
    problems = problems[:n_problems]
    
    print(f"Loaded {len(problems)} problems.")
    print(f"\nExample:")
    print(f"  Q: {problems[0]['question'][:120]}...")
    print(f"  A: {problems[0]['ground_truth']}")
    return problems

problems = load_gsm8k(NUM_PROBLEMS)

In [ ]:
def load_model_fp16(model_id):
    print(f"Loading {model_id}...")
    local_path = "./qwen3b" if "3B" in model_id else model_id
    
    tokenizer = AutoTokenizer.from_pretrained(local_path)
    print("  Tokenizer loaded ✓")
    
    print("  Loading weights (this takes 2-3 min, no progress bar)...")
    model = AutoModelForCausalLM.from_pretrained(
        local_path,
        torch_dtype=torch.float16,
    )
    print("  Weights loaded, moving to MPS...")
    model = model.to("mps")
    model.eval()
    
    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"  Parameters : {n_params:.1f}B")
    print("  Done ✓")
    return model, tokenizer

In [ ]:
# ============================================================
# CELL 6: Prompt, Generation & Answer Extraction
# ============================================================

def build_prompt(question, tokenizer):
    messages = [
        {"role": "system", "content": "You are a math tutor. Solve step by step. End with your final answer as: #### <number>"},
        {"role": "user", "content": question}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def extract_answer(text):
    """Pull the final numeric answer from model output."""
    match = re.search(r'####\s*([\d,\.]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    numbers = re.findall(r'\b(\d+(?:\.\d+)?)\b', text)
    return numbers[-1] if numbers else None


def majority_vote(answers):
    """Self-Consistency: pick the most common valid answer."""
    valid = [a for a in answers if a is not None]
    if not valid:
        return None
    return Counter(valid).most_common(1)[0][0]


def is_correct(predicted, ground_truth):
    if predicted is None:
        return False
    try:
        return abs(float(predicted) - float(ground_truth)) < 1e-6
    except:
        return str(predicted).strip() == str(ground_truth).strip()


def generate_n_answers(model, tokenizer, question, n=1, temperature=0.7):
    """
    Generate N independent answers in one batched forward pass.
    temperature > 0 creates diversity between samples (essential for voting).
    """
    prompt = build_prompt(question, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]

    start = time.time()
    with torch.no_grad():
        if n == 1:
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            responses = [tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)]
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=temperature,
                num_return_sequences=n,
                pad_token_id=tokenizer.eos_token_id
            )
            responses = [
                tokenizer.decode(outputs[i][input_len:], skip_special_tokens=True)
                for i in range(n)
            ]

    elapsed = time.time() - start
    answers = [extract_answer(r) for r in responses]
    return responses, answers, elapsed

In [ ]:
# ============================================================
# CELL 7: Evaluation Loop
# ============================================================

def evaluate_model(model, tokenizer, problems, n_values, model_name, save_path=None):
    """
    Sweep N values. For each N:
      1. Generate N answers per problem
      2. Majority vote -> final answer
      3. Compare to ground truth
      4. Record accuracy + timing
    JSON checkpoint saved after each N so progress is never lost.
    """
    all_results = {}

    for n in n_values:
        print(f"\n{'='*55}")
        print(f"  {model_name}  |  N = {n}")
        print(f"{'='*55}")

        correct = 0
        total_time = 0.0
        problem_results = []

        for i, prob in enumerate(tqdm(problems, desc=f"N={n}")):
            try:
                responses, answers, elapsed = generate_n_answers(
                    model, tokenizer, prob['question'],
                    n=n, temperature=TEMPERATURE
                )
                final = majority_vote(answers)
                ok    = is_correct(final, prob['ground_truth'])

                correct    += int(ok)
                total_time += elapsed

                problem_results.append({
                    'ground_truth': prob['ground_truth'],
                    'answers'     : answers,
                    'final_answer': final,
                    'correct'     : ok,
                    'time'        : elapsed
                })

                if (i + 1) % 25 == 0:
                    print(f"  [{i+1}/{len(problems)}] running acc: {correct/(i+1)*100:.1f}%")

            except Exception as e:
                print(f"  Problem {i} error: {e}")
                problem_results.append({'correct': False, 'error': str(e)})

        accuracy = correct / len(problems) * 100
        avg_time = total_time / len(problems)

        all_results[n] = {
            'accuracy'            : accuracy,
            'avg_time_per_problem': avg_time,
            'total_correct'       : correct,
            'n_problems'          : len(problems),
            'problem_results'     : problem_results
        }

        print(f"  Accuracy        : {accuracy:.1f}%")
        print(f"  Avg time/problem: {avg_time:.1f}s")

        if save_path:
            safe = {k: {kk: vv for kk, vv in v.items() if kk != 'problem_results'}
                    for k, v in all_results.items()}
            with open(save_path, 'w') as f:
                json.dump(safe, f, indent=2)

    return all_results

In [ ]:
print('HI')

In [ ]:
# ============================================================
# CELL 8: Smoke Test — run this before the full experiment!
# ============================================================

small_model, small_tokenizer = load_model_fp16(SMALL_MODEL_ID)

test_q = "Janet has 5 apples. She gives 2 to her friend. How many does she have left?"
resps, answers, t = generate_n_answers(small_model, small_tokenizer, test_q, n=3)

print(f"\nAnswers      : {answers}")
print(f"Majority vote: {majority_vote(answers)}")
print(f"Time         : {t:.1f}s")
print("\nSmoke test passed ✓")

In [ ]:
# ============================================================
# CELL 9: Run Small Model — Full Best-of-N Sweep
# ============================================================
# NUM_PROBLEMS=50  -> ~30-60 min
# NUM_PROBLEMS=300 -> ~3-6 hours (set this for your final paper run)

small_results = evaluate_model(
    small_model, small_tokenizer,
    problems,
    n_values=N_VALUES,
    model_name="Qwen2.5-3B fp16",
    save_path="results_small.json"
)
print("\nSmall model run complete.")

In [ ]:
# ============================================================
# CELL 10: Load Teacher Model
# ============================================================

del small_model
torch.cuda.empty_cache()
print(f"Small model freed. VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

teacher_model, teacher_tokenizer = load_model_fp16(TEACHER_MODEL_ID, max_gpu_gb="12GiB")

In [ ]:
# ============================================================
# CELL 11: Run Teacher Model — N=1 baseline only
# ============================================================

teacher_results = evaluate_model(
    teacher_model, teacher_tokenizer,
    problems,
    n_values=[1],
    model_name="Qwen2.5-7B fp16",
    save_path="results_teacher.json"
)

TEACHER_ACC  = teacher_results[1]['accuracy']
TEACHER_TIME = teacher_results[1]['avg_time_per_problem']
print(f"\nTeacher accuracy: {TEACHER_ACC:.1f}%")
print(f"Teacher avg time: {TEACHER_TIME:.1f}s")

In [ ]:
# ============================================================
# CELL 12: Results Table
# ============================================================

print(f"{'='*62}")
print(f"{'Configuration':<30} {'Acc':>8} {'Time':>8} {'vs Teacher':>12}")
print(f"{'-'*62}")
for n in N_VALUES:
    acc  = small_results[n]['accuracy']
    t    = small_results[n]['avg_time_per_problem']
    gap  = acc - TEACHER_ACC
    sign = f"+{gap:.1f}%" if gap >= 0 else f"{gap:.1f}%"
    print(f"  Qwen2.5-3B  N={n:<3}          {acc:>7.1f}% {t:>7.1f}s {sign:>12}")
print(f"{'-'*62}")
print(f"  Qwen2.5-7B  N=1 (teacher)  {TEACHER_ACC:>7.1f}% {TEACHER_TIME:>7.1f}s {'(target)':>12}")
print(f"{'='*62}")

In [ ]:
# ============================================================
# CELL 13: Plots
# ============================================================

accs  = [small_results[n]['accuracy']             for n in N_VALUES]
times = [small_results[n]['avg_time_per_problem'] for n in N_VALUES]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Test-Time Scaling: Best-of-N on GSM8K', fontsize=13, fontweight='bold')

# Plot 1: Accuracy vs N
ax = axes[0]
ax.plot(N_VALUES, accs, 'o-', color='#c8401a', lw=2, ms=8, label='Small 3B + Best-of-N')
ax.axhline(TEACHER_ACC, color='#1a4fc8', ls='--', lw=2, label=f'Teacher 7B N=1 ({TEACHER_ACC:.1f}%)')
crossover = next((n for n, a in zip(N_VALUES, accs) if a >= TEACHER_ACC), None)
if crossover:
    ax.axvline(crossover, color='green', ls=':', lw=1.5, label=f'Crossover N={crossover}')
ax.set_xlabel('N (samples)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs. N'); ax.legend(fontsize=8)
ax.set_xticks(N_VALUES); ax.grid(alpha=0.3)

# Plot 2: Accuracy-Latency Curve
ax = axes[1]
ax.plot(times, accs, 'o-', color='#c8401a', lw=2, ms=8, label='Small 3B')
ax.axhline(TEACHER_ACC, color='#1a4fc8', ls='--', lw=2, label='Teacher 7B')
ax.axvline(TEACHER_TIME, color='#1a4fc8', ls=':', alpha=0.5)
for n, t, a in zip(N_VALUES, times, accs):
    ax.annotate(f'N={n}', (t, a), textcoords='offset points', xytext=(5, 4), fontsize=8)
ax.set_xlabel('Avg Time / Problem (s)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy-Latency Tradeoff'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Plot 3: Marginal Gain per N
ax = axes[2]
baseline = accs[0]
gains  = [a - baseline for a in accs]
colors = ['#166534' if g >= (TEACHER_ACC - baseline) else '#c8401a' for g in gains]
bars   = ax.bar(N_VALUES, gains, color=colors, alpha=0.75)
ax.axhline(TEACHER_ACC - baseline, color='#1a4fc8', ls='--', lw=1.5,
           label=f'Gap to close: {TEACHER_ACC-baseline:.1f}%')
for bar, g in zip(bars, gains):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'+{g:.1f}%', ha='center', fontsize=8)
ax.set_xlabel('N'); ax.set_ylabel('Accuracy Gain over N=1 (%)')
ax.set_title('Marginal Gain from More Samples')
ax.legend(fontsize=8); ax.set_xticks(N_VALUES); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved results_plots.png")

In [ ]:
# ============================================================
# CELL 14: Save Final Results
# ============================================================

final = {
    'config': {
        'small_model'  : SMALL_MODEL_ID,
        'teacher_model': TEACHER_MODEL_ID,
        'n_problems'   : NUM_PROBLEMS,
        'n_values'     : N_VALUES,
        'temperature'  : TEMPERATURE,
        'dataset'      : 'GSM8K',
        'precision'    : 'float16'
    },
    'small_model': {
        n: {
            'accuracy': small_results[n]['accuracy'],
            'avg_time': small_results[n]['avg_time_per_problem']
        } for n in N_VALUES
    },
    'teacher': {
        'accuracy': TEACHER_ACC,
        'avg_time': TEACHER_TIME
    }
}

with open('final_results.json', 'w') as f:
    json.dump(final, f, indent=2)

print("Saved final_results.json")
print("Saved results_plots.png")
print("All done!")